In [1]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql import functions as F

# Pacotes necessários (incluindo Delta)
packages = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",
    "org.apache.hadoop:hadoop-aws:3.3.2",
    "com.amazonaws:aws-java-sdk-bundle:1.12.628"
])

builder = SparkSession.builder \
    .appName("DeltaLakeApp") \
    .config("spark.jars.packages", packages) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"✅ Spark {spark.version} com Delta Lake configurado!")

✅ Spark 3.5.0 com Delta Lake configurado!


In [3]:
spark

In [47]:
nam_path_s3 = "s3a://datalake/tmp_data/netflix_titles/"

#### Criando tabela em delta

In [4]:
df_pyspark=spark.read.csv('s3a://datalake/raw_data/netflix_titles.csv', inferSchema=True, header=True, sep=',', quote='"', escape='"', multiLine=True)

In [49]:
df_pyspark.write.mode("overwrite").format("delta").option("path",nam_path_s3).saveAsTable('netflix_titles')

#### Leitura de dados

In [50]:
df_delta  = spark.read.format("delta").load(nam_path_s3)

In [23]:
df_delta.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [24]:
df_delta.groupBy('type').agg(F.count('*')).show(10)

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  Movie|    6131|
+-------+--------+



### Update dos dados

In [51]:
table = DeltaTable.forPath(spark,nam_path_s3)

In [26]:
table.update(
    condition="type = 'Movie'",
    set={"type": "'movie'"})

In [42]:
table.toDF().groupBy('type').agg(F.count('*')).show()

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  movie|    6131|
+-------+--------+



### Delete dos Dados

In [61]:
table_delete = DeltaTable.forPath(spark,nam_path_s3)
table_delete.delete("show_id='s1'")

In [62]:
table_delete.toDF().select('show_id','title').filter(F.col('show_id')=='s1').show(10)

+-------+-----+
|show_id|title|
+-------+-----+
+-------+-----+



In [63]:
table_delete.toDF().select('show_id','title').show(10)

+-------+--------------------+
|show_id|               title|
+-------+--------------------+
|     s2|       Blood & Water|
|     s3|           Ganglands|
|     s4|Jailbirds New Orl...|
|     s5|        Kota Factory|
|     s6|       Midnight Mass|
|     s7|My Little Pony: A...|
|     s8|             Sankofa|
|     s9|The Great British...|
|    s10|        The Starling|
|    s11|Vendetta: Truth, ...|
+-------+--------------------+
only showing top 10 rows

